# 그룹화와 집계 (Groupby & aggregate)

#### 핵심 개념
1. groupby(): 같은 종류끼리 모아놓기(반찬통에 김, 우유, 달걀 따로따로 담기)
2. 집계함수: 모아놓은 것들을 계산하기 (달걀이 몇 개? 우유가 몇 개?)
3. agg(): 여러 개 계산을 한 번에 하기
4. fillter(): 조건에 맞는 것만 고르기
5. 다중 그룹화: 2가지 이상으로 나누기 (나라별 + 도시별)

In [2]:
# 1단계: groupby() - 데이터 그룹화

import pandas as pd

# 샘플 데이터 (마케팅 캠페인 데이터)

df = pd.DataFrame({
    'product': ['노트북', '마우스', '노트북', '마우스', '키보드'],
    'sales': [100, 50, 150, 60, 80],
    'data': ['2024-01-01', '2024-01-01', '2024-01-02', '2024-01-02', '2024-01-03']
})

# 제품별로 그룹화하기
grouped = df.groupby('product')
print(grouped) # 그룹화된 객체 보기

# 각 그룹 따로따로 보기
for name, group in grouped:
    print(f"\n=== {name} ===")
    print(group)


=== 노트북 ===
  product  sales        data
0     노트북    100  2024-01-01
2     노트북    150  2024-01-02

=== 마우스 ===
  product  sales        data
1     마우스     50  2024-01-01
3     마우스     60  2024-01-02

=== 키보드 ===
  product  sales        data
4     키보드     80  2024-01-03


In [3]:
# 2단계: 집계 함수 sum(), mean(), count()

# 제품별 매출 합계
print(df.groupby('product')['sales'].sum())

# 제품별 평균 매출
print(df.groupby('product')['sales'].mean())

# 제품별 데이터 개수
print(df.groupby('product')['sales'].count())

product
노트북    250
마우스    110
키보드     80
Name: sales, dtype: int64
product
노트북    125.0
마우스     55.0
키보드     80.0
Name: sales, dtype: float64
product
노트북    2
마우스    2
키보드    1
Name: sales, dtype: int64


In [4]:
# 자주 쓰는 다양한 집계 함수들
grouped['sales'].max() # 최대값
grouped['sales'].min() # 최소값
grouped['sales'].std() # 표준편차
grouped['sales'].first() # 첫 번째 값
grouped['sales'].last() # 마지막 값

product
노트북    150
마우스     60
키보드     80
Name: sales, dtype: int64

In [ ]:
# 3단계: agg() 여러 함수 동시 적용

# 방법1) 리스트로 여러 함수 적용
result = df.groupby('product')['sales'].agg(['sum', 'mean', 'count'])
print(result)

         sum   mean  count
product                   
노트북      250  125.0      2
마우스      110   55.0      2
키보드       80   80.0      1


In [6]:
# 방법2) 딕셔너리로 컬럼별로 다른 함수 적용

# 각 컬럼마다 다른 함수를 적용하고 싶을 때
result = df.groupby('product').agg({
    'sales': ['sum', 'mean', 'count'],
    'data': 'count' # data는 개수만 셈
})

print(result)

        sales               data
          sum   mean count count
product                         
노트북       250  125.0     2     2
마우스       110   55.0     2     2
키보드        80   80.0     1     1


In [7]:
# 방법3) 함수 이름 커스터마이징

result = df.groupby('product')['sales'].agg(
    총합=('sum'),
    평균=('mean'),
    개수=('count')
)
print(result)

# 또는 이렇게 (더 깔씀한 방식)
result = df.groupby('product')['sales'].agg(
    total='sum',
    average='mean',
    count='count'
)
print(result)

          총합     평균  개수
product                
노트북      250  125.0   2
마우스      110   55.0   2
키보드       80   80.0   1
         total  average  count
product                       
노트북        250    125.0      2
마우스        110     55.0      2
키보드         80     80.0      1


In [8]:
# 4단계: fullter() 그룹별 필터링

# 매출 합계가 150 이상인 제품만 보고 싶을 땐
result = df.groupby('product')['sales'].filter(lambda x: x.sum() >= 150)
print(result)

# 더 명확하게 보려면
result = df.groupby('product')['sales'].transform('sum') >= 150
print(result)

0    100
2    150
Name: sales, dtype: int64
0     True
1    False
2     True
3    False
4    False
Name: sales, dtype: bool


In [10]:
# 예시: 평균 매출이 70이상인 제품 데이터만

# 각 제품의 평균 매출을 계산하고
product_avg = df.groupby('product')['sales'].mean()

# 평균이 70 이상인 제품들
high_performers = product_avg[product_avg >= 70]
print(high_performers)

# 그 제품들의 모든 데이터 보기
result = df[df['product'].isin(high_performers.index)]
print(result)

product
노트북    125.0
키보드     80.0
Name: sales, dtype: float64
  product  sales        data
0     노트북    100  2024-01-01
2     노트북    150  2024-01-02
4     키보드     80  2024-01-03


In [11]:
# 5단계: 다중 레벨 그룹화 / 2가지 기준으로 그룹화하기

# 더 자세한 데이터셋 만들기
df2 = pd.DataFrame({
    '지역': ['서울', '서울', '부산', '부산', '서울', '부산'],
    'product': ['노트북', '마우스', '노트북', '마우스', '마우스', '노트북'],
    'sales': [100, 50, 80, 60, 45, 120]
})

# 지역과 제품으로 함께 그룹화
result = df2.groupby(['지역', 'product'])['sales'].agg(['sum', 'mean', 'count'])
print(result)

            sum   mean  count
지역 product                   
부산 노트북      200  100.0      2
   마우스       60   60.0      1
서울 노트북      100  100.0      1
   마우스       95   47.5      2


In [12]:
# 3가지 기분으로 그룹화하기 (더 복잡한 분석)
df3 = pd.DataFrame({
    '연도': [2024, 2024, 2024, 2024, 2024],
    '월': [1, 1, 2, 2, 2],
    'product': ['노트북', '마우스', '노트북', '마우스', '키보드'],
    'sales': [100, 50, 150, 60, 80]
})

result = df3.groupby(['연도', '월', 'product'])['sales'].sum()
print(result)

연도    월  product
2024  1  노트북        100
         마우스         50
      2  노트북        150
         마우스         60
         키보드         80
Name: sales, dtype: int64


In [15]:
# 예시: 마케팅 분석

# 마케팅 캠체인 데이터
campaign_df = pd.DataFrame({
    'campaign': ['이메일', '이메일', '소셜', '소셜', '이메일', '소셜'],
    'day': ['월', '화', '월', '화', '수', '수'],
    'clicks': [100, 120, 80, 90, 110, 95],
    'conversions': [10, 12, 8, 9, 11, 10]
})

# 캠페인 성과 분석
print("====== 캠페인별 성과 =====")
result = campaign_df.groupby('campaign').agg({
    'clicks': ['sum', 'mean'],
    'conversions': ['sum', 'mean']
})
print(result)

# 캠페인과 요일별 분석
print("\n===== 캠페인 x 요일별 성과 =====")
result = campaign_df.groupby(['campaign', 'day'])['clicks'].sum()
print(result)

# 전환율이 높은 캠페인만 보기
campaign_df['conversion_rate'] = campaign_df['conversions'] / campaign_df['clicks']
high_rate = campaign_df.groupby('campaign')['conversion_rate'].mean()
print("\n===== 전환율 =====")
print(high_rate)

====== 캠페인별 성과 =====
         clicks             conversions      
            sum        mean         sum  mean
campaign                                     
소셜          265   88.333333          27   9.0
이메일         330  110.000000          33  11.0

===== 캠페인 x 요일별 성과 =====
campaign  day
소셜        수       95
          월       80
          화       90
이메일       수      110
          월      100
          화      120
Name: clicks, dtype: int64

===== 전환율 =====
campaign
소셜     0.101754
이메일    0.100000
Name: conversion_rate, dtype: float64
